# Lesson 13 · Gravitational fields

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/13_gravitational_fields.ipynb)

**For:** A-level Physics (for example AQA 3.7.2, OCR A 5.4, Edexcel Topic 12) and IB Physics (D.1 gravitational fields). Check your own specification for the exact wording.

**Before this lesson:** Lesson 3, *Circular motion and gravitational orbits* (circular motion, centripetal force and orbits); Lesson 5, *Electric fields and potential* (field strength and potential, which gravity mirrors). Also assumed: Newton's laws of motion and gravitational potential energy near the Earth.

**Leads to:** the end of this route for now.

**You will be able to:**

1. use Newton's law of gravitation, $F = GMm/r^2$, and the field strength $g = GM/r^2$
2. add the fields of several masses as vectors, and find a point where they cancel
3. explain gravitational potential, $V = -GM/r$, and link it to the field by $g = -\Delta V/\Delta r$
4. derive the speed and period of a circular orbit, including Kepler's third law $T^2 \propto r^3$
5. find the escape speed $v = \sqrt{2GM/r}$ from an energy argument

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

[Launch from a planet: try it in the explorer](../explorer/#launch).

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import G          # 6.674e-11 N m^2 kg^-2
from qplay import gravity

M_E, R_E = 5.97e24, 6.371e6            # Earth: mass in kg, radius in m
M_moon, R_moon = 7.35e22, 1.737e6      # Moon: mass in kg, radius in m
d_moon = 3.84e8                        # Earth-Moon distance in m

## 1. Newton's law of gravitation and field strength

Every pair of masses attracts with a force along the line joining them,

$$F = \frac{GMm}{r^2},$$

where $r$ is the distance between their centres and $G = 6.67\times10^{-11}\ \mathrm{N\,m^2\,kg^{-2}}$. A spherical planet pulls as if all its mass were at its centre.

The **gravitational field strength** at a point is the force per unit mass on a small test mass placed there,

$$g = \frac{F}{m} = \frac{GM}{r^2}\qquad(\text{unit: N kg}^{-1}\text{, the same as m s}^{-2}).$$

It is a vector, pointing towards the mass. `gravity.field(point, masses, positions)` adds up $-GM\,\mathbf r/r^3$ for every mass and returns the field vector in m s$^{-2}$.

In [ ]:
r = np.linspace(R_E, 10 * R_E, 400)
g = np.array([np.linalg.norm(gravity.field([x, 0], [M_E], [[0, 0]])) for x in r])

plt.figure(figsize=(7, 3.4))
plt.plot(r / R_E, g, label="gravity.field")
plt.plot(r / R_E, 9.82 / (r / R_E)**2, "k:", label=r"$g_0 (R_E/r)^2$")
plt.xlabel("distance from Earth's centre / Earth radii"); plt.ylabel("g / N kg$^{-1}$")
plt.legend(); plt.title("Earth's field falls as the inverse square of distance")
print(f"g at the surface: {g[0]:.2f} N/kg;  at 2 R_E: {np.interp(2 * R_E, r, g):.2f} N/kg;  at 10 R_E: {g[-1]:.3f} N/kg")

Doubling the distance from the centre quarters the field; ten times as far, it is a hundred times weaker.

## 2. Adding fields: the Earth and the Moon

When several masses pull on the same point, their fields add as **vectors**. Between the Earth and the Moon the two fields point in opposite directions, so somewhere along the line they cancel. A spacecraft coasting there would feel no net pull.

**Predict (P1).** Where on the line from the Earth to the Moon is the total field zero: halfway, nearer the Earth, or nearer the Moon? Roughly what fraction of the way is it? Write your prediction down before you run the next cell.

In [ ]:
x = np.linspace(0.7 * d_moon, 0.98 * d_moon, 2000)
g_x = np.array([gravity.field([xi, 0], [M_E, M_moon], [[0, 0], [d_moon, 0]])[0] for xi in x])
neutral = x[np.argmin(np.abs(g_x))]

plt.figure(figsize=(7, 3.4))
plt.plot(x / d_moon, 1e3 * g_x)
plt.axhline(0, color="0.7", lw=0.6)
plt.axvline(neutral / d_moon, color="C3", ls="--", label=f"field zero at {neutral / d_moon:.3f} of the way")
plt.ylim(-3, 3)
plt.xlabel("distance from the Earth / Earth–Moon distance"); plt.ylabel("x-component of g / mN kg$^{-1}$")
plt.legend(); plt.title("Total field on the Earth–Moon line (negative: towards the Earth)")
print(f"field is zero {neutral:.3e} m from the Earth, {d_moon - neutral:.3e} m from the Moon")
print(f"check: d / (1 + sqrt(m/M)) = {d_moon / (1 + np.sqrt(M_moon / M_E)):.3e} m")

**Explain (P1).** The Moon has about $1/81$ of the Earth's mass. Use $g = GM/r^2$ to explain why the neutral point is where it is.

## 3. Gravitational potential

The **gravitational potential** $V$ at a point is the work done per unit mass to bring a small mass from infinity to that point. Gravity pulls the mass in, so that work is negative, and

$$V = -\frac{GM}{r}\qquad(\text{unit: J kg}^{-1}).$$

The zero of potential is at infinity, so every potential near a mass is negative, and the deeper you are in the field the more negative it gets. Moving a mass $m$ between two points changes its potential energy by $m\,\Delta V$.

Field and potential are linked: the field is minus the **gradient** of the potential,

$$g = -\frac{\Delta V}{\Delta r},$$

so the field is strongest where the potential graph is steepest. The next cell checks this numerically.

In [ ]:
r = np.linspace(R_E, 6 * R_E, 400)
V = np.array([gravity.potential([x, 0], [M_E], [[0, 0]]) for x in r])
g_from_V = -np.gradient(V, r)                           # minus the slope of V(r)
g_direct = np.array([gravity.field([x, 0], [M_E], [[0, 0]])[0] for x in r])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(r / R_E, V / 1e6)
ax1.set_xlabel("r / Earth radii"); ax1.set_ylabel("V / MJ kg$^{-1}$"); ax1.set_title(r"$V = -GM/r$")
ax2.plot(r / R_E, -g_direct, label="|gravity.field|")
ax2.plot(r[::25] / R_E, -g_from_V[::25], "o", ms=4, label=r"$-\Delta V/\Delta r$, numerically")
ax2.set_xlabel("r / Earth radii"); ax2.set_ylabel("g / N kg$^{-1}$"); ax2.legend(); ax2.set_title("Field from the gradient of V")
print(f"potential at the surface: {V[0] / 1e6:.1f} MJ/kg")
inner = slice(1, -1)                                    # np.gradient is one-sided, so less accurate, at the two ends
print(f"largest relative difference between the two field curves: {np.max(np.abs(g_from_V[inner] - g_direct[inner]) / np.abs(g_direct[inner])):.1e}")

**Predict (P2).** A probe sits exactly halfway between two identical planets, each of mass $5.97\times10^{24}$ kg and 8000 km from the probe. The fields of the two planets cancel there. Does that mean the potential is zero too?

In [ ]:
m, r = 5.97e24, 8.0e6
positions = [[-r, 0], [r, 0]]
print("field at the midpoint:", np.round(gravity.field([0, 0], [m, m], positions), 12) + 0.0, "N/kg")
print(f"potential at the midpoint: {gravity.potential([0, 0], [m, m], positions):.3e} J/kg")
print(f"potential from one planet alone: {gravity.potential([0, 0], [m], [[-r, 0]]):.3e} J/kg")

**Explain (P2).** Field is a vector and potential is a scalar. Explain why one adds to zero here and the other does not.

## 4. Orbits and escape

For a satellite in a circular orbit of radius $r$, gravity provides the centripetal force:

$$\frac{GMm}{r^2} = \frac{mv^2}{r}\quad\Rightarrow\quad v = \sqrt{\frac{GM}{r}},\qquad T = \frac{2\pi r}{v}\quad\Rightarrow\quad T^2 = \frac{4\pi^2}{GM}r^3 .$$

That last result is **Kepler's third law**: the square of the period is proportional to the cube of the orbit radius, with the same constant for everything orbiting the same body. The satellite's mass cancels, so a spanner and a space station share an orbit.

A **geostationary** satellite orbits above the equator once per sidereal day (86 164 s), so it stays above one point on the ground. `gravity.synchronous_radius(M)` solves Kepler's law for that radius.

In [ ]:
T_moon = 2 * np.pi * np.sqrt(d_moon**3 / (G * M_E))
print(f"Moon's orbital period from Kepler's third law: {T_moon / 86400:.1f} days (measured: 27.3 days)")

r_geo = gravity.synchronous_radius(M_E)
print(f"geostationary orbit: radius {r_geo / 1e3:,.0f} km, altitude {(r_geo - R_E) / 1e3:,.0f} km, "
      f"speed {np.sqrt(G * M_E / r_geo):,.0f} m/s")

r = np.geomspace(R_E, 2 * d_moon, 200)
T = 2 * np.pi * np.sqrt(r**3 / (G * M_E))
plt.figure(figsize=(6, 4))
plt.loglog(r / 1e3, T / 3600)
for name, radius in [("low orbit", R_E + 400e3), ("geostationary", r_geo), ("Moon", d_moon)]:
    period = 2 * np.pi * np.sqrt(radius**3 / (G * M_E))
    plt.plot(radius / 1e3, period / 3600, "o", color="C3")
    plt.annotate(name, (radius / 1e3, period / 3600), textcoords="offset points", xytext=(-10, 8), ha="right")
plt.xlabel("orbit radius / km"); plt.ylabel("period / hours")
plt.title(r"Kepler's third law: slope 3/2 on log axes");

To **escape** completely, a projectile needs enough kinetic energy to climb from potential $-GM/r$ all the way to $0$ at infinity:

$$\tfrac12 mv^2 = \frac{GMm}{r}\quad\Rightarrow\quad v_{\text{esc}} = \sqrt{\frac{2GM}{r}} .$$

From the Earth's surface that is about 11.2 km s$^{-1}$, ignoring air resistance.

**Predict (P3).** A spacecraft is already 4 times as far from the Earth's centre as the surface is. Compared with escaping from the surface, what is its escape speed?

In [ ]:
for factor in (1, 4):
    print(f"escape speed from {factor} x R_E: {gravity.escape_speed(M_E, factor * R_E):,.0f} m/s")

**Explain (P3).** Use $v_{\text{esc}} = \sqrt{2GM/r}$ to explain the ratio.

**Common mistakes**

- Measuring $r$ from the planet's surface. In $GM/r^2$ and $GM/r$, $r$ is measured from the **centre**: add the radius to the altitude.
- Forgetting the minus sign in potential. Potential is zero at infinity and negative everywhere else.
- Adding potentials as vectors, or fields as plain numbers. Fields are vectors; potentials are scalars.
- Thinking a satellite needs fuel to stay in a circular orbit. Gravity supplies the centripetal force; no thrust is needed.

---

## Exercises

Take $G = 6.67\times10^{-11}\ \mathrm{N\,m^2\,kg^{-2}}$, Earth's mass $5.97\times10^{24}$ kg and radius 6371 km. Give answers to 3 significant figures.

**Q1.** Find the gravitational field strength at the height of the International Space Station, 400 km above the Earth's surface.

**Q2.** Find the orbital period of the Space Station, in minutes.

**Q3.** Find the energy needed to lift a 1000 kg satellite from the Earth's surface to a height of 400 km (ignore its orbital speed): use the change in potential.

**Q4.** Find the escape speed from the surface of the Moon (mass $7.35\times10^{22}$ kg, radius 1737 km).

**Q5.** Find the altitude, in km, of a geostationary satellite above the Earth's surface. Use a sidereal day of 86 164 s.

**Q6 (simulation).** Using `gravity.field`, find the distance from the Earth's centre of the point between the Earth and the Moon where their fields cancel. Take the Moon's mass as $7.35\times10^{22}$ kg and the Earth–Moon distance as $3.84\times10^{8}$ m.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_r = R_E + 400e3
_answers = {
    "Q1": np.linalg.norm(gravity.field([_r, 0], [M_E], [[0, 0]])),
    "Q2": 2 * np.pi * np.sqrt(_r**3 / (G * M_E)) / 60,
    "Q3": 1000 * (gravity.potential([_r, 0], [M_E], [[0, 0]]) - gravity.potential([R_E, 0], [M_E], [[0, 0]])),
    "Q4": gravity.escape_speed(M_moon, R_moon),
    "Q5": (gravity.synchronous_radius(M_E) - R_E) / 1e3,
    "Q6": d_moon / (1 + np.sqrt(M_moon / M_E)),
}

# Replace None with your answers, in SI units unless the question says otherwise.
mine = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** $r = 6371 + 400 = 6771$ km. $g = \dfrac{GM}{r^2} = \dfrac{6.67\times10^{-11} \times 5.97\times10^{24}}{(6.771\times10^6)^2} = 8.69\ \mathrm{N\,kg^{-1}}$. Astronauts float not because gravity is absent (it is still about 89% of its surface value) but because they and the station are falling together.

**Q2.** $T = 2\pi\sqrt{\dfrac{r^3}{GM}} = 2\pi\sqrt{\dfrac{(6.771\times10^6)^3}{6.67\times10^{-11} \times 5.97\times10^{24}}} = 5546$ s, which is 92.4 minutes.

**Q3.** $\Delta V = -\dfrac{GM}{r_2} - \left(-\dfrac{GM}{r_1}\right) = GM\left(\dfrac{1}{6.371\times10^6} - \dfrac{1}{6.771\times10^6}\right) = 3.69\times10^{6}\ \mathrm{J\,kg^{-1}}$, so the energy is $m\,\Delta V = 3.69\times10^{9}$ J. (Using $mg\Delta h$ with $g = 9.82$ gives $3.93\times10^9$ J, too much, because $g$ weakens with height.)

**Q4.** $v = \sqrt{\dfrac{2GM}{r}} = \sqrt{\dfrac{2 \times 6.67\times10^{-11} \times 7.35\times10^{22}}{1.737\times10^6}} = 2.38\times10^{3}\ \mathrm{m\,s^{-1}}$, about 2.38 km/s.

**Q5.** $r = \left(\dfrac{GMT^2}{4\pi^2}\right)^{1/3} = 42159$ km from the centre, so the altitude is $42159 - 6371 = 35788$ km, about 35 800 km.

**Q6.** At distance $x$ from the Earth the fields cancel when $\dfrac{GM}{x^2} = \dfrac{Gm}{(d - x)^2}$, so $\dfrac{d - x}{x} = \sqrt{\dfrac{m}{M}}$ and $x = \dfrac{d}{1 + \sqrt{m/M}} = 3.46\times10^{8}$ m, about 90% of the way to the Moon. The scan below agrees.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}".rstrip("."))

_x = np.linspace(0.8 * d_moon, 0.95 * d_moon, 30001)
_gx = np.array([gravity.field([xi, 0], [M_E, M_moon], [[0, 0], [d_moon, 0]])[0] for xi in _x])
print(f"scan: the field changes sign at {_x[np.argmax(_gx > 0)]:.4e} m from the Earth")

### Predict and explain: answers

**P1.** Nearer the Moon, about 90% of the way there. The Moon's field is weaker because its mass is about 81 times smaller, so the probe must be about $\sqrt{81} = 9$ times closer to the Moon than to the Earth for the two fields to match: $9/10$ of the way along the line.

**P2.** No. The fields are equal in size and opposite in direction, so as vectors they cancel. Potential has no direction: each planet contributes a negative potential, $-GM/r$, and two negatives add to a larger negative. The probe sits at the top of a potential "ridge", not at zero.

**P3.** It halves. Escape speed is proportional to $1/\sqrt r$, so four times the distance gives $1/\sqrt4 = 1/2$ of the speed: about 5.6 km/s instead of 11.2 km/s. The spacecraft is already a quarter of the way up the potential well, in potential terms, so it needs a quarter of the kinetic energy per kilogram.